# Continuous Polarization-BSM Dataset Validation
Structural and physics checks. Hidden truth is used only for validation.


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATES = [
    Path('generated/polarization_bsm_60s'),
    Path('example/entanglement_swapping_validation/ai_bsm_dataset/generated/polarization_bsm_60s'),
]
RESULTS_DIR = next((path for path in CANDIDATES if (path / 'metadata.json').exists()), CANDIDATES[-1])
metadata = json.loads((RESULTS_DIR / 'metadata.json').read_text(encoding='utf-8'))
observable = pd.read_csv(RESULTS_DIR / 'bsm_observable_windows.csv')
labels = pd.read_csv(RESULTS_DIR / 'episode_labels.csv')
truth = pd.read_csv(RESULTS_DIR / 'hidden_truth.csv')
metadata


{'dataset_name': 'polarization_bsm_60s',
 'config_signature': '1e312217f4c1c1e325d9e75664bee492456f3f74b9f18e13184278b391109c9d',
 'episodes': 10000,
 'windows': 600000,
 'episode_duration_s': 60.0,
 'window_duration_s': 1.0,
 'continuous_timeline_per_episode': True,
 'fault_labels': ['healthy',
  'temperature_change',
  'polarization_drift',
  'spectral_detuning',
  'raman_noise',
  'attenuation_loss',
  'source_brightness_loss',
  'sync_issue',
  'source_clock_drift'],
 'ai_input_file': 'bsm_observable_windows.csv',
 'note': 'Only four BSM detector streams are model inputs; hidden truth is offline only.'}

In [3]:
checks = {
    'continuous timeline declared': metadata.get('continuous_timeline_per_episode') is True,
    'episode count': observable['episode_id'].nunique() == metadata['episodes'],
    'window count': len(observable) == metadata['windows'],
    'four detector rates': all(f'{d}_rate_hz' in observable for d in ['d3h','d3v','d4h','d4v']),
    'no simulator truth in inputs': not any(c.startswith('true_') for c in observable),
    'unique episode/window': not observable.duplicated(['episode_id','window_index']).any(),
}
pd.Series(checks, name='passed')


continuous timeline declared    True
episode count                   True
window count                    True
four detector rates             True
no simulator truth in inputs    True
unique episode/window           True
Name: passed, dtype: bool

In [4]:
expected = round(metadata['episode_duration_s'] / metadata['window_duration_s'])
window_counts = observable.groupby('episode_id').size()
print('Expected windows per episode:', expected)
print(window_counts.describe())
assert checks['continuous timeline declared']
assert all(window_counts == expected)
assert all(checks.values())


Expected windows per episode: 60
count    10000.0
mean        60.0
std          0.0
min         60.0
25%         60.0
50%         60.0
75%         60.0
max         60.0
dtype: float64


In [5]:
summary = observable.groupby('fault_class')[[
    'total_singles_rate_hz','accepted_bsm_rate_hz','psi_minus_rate_hz',
    'psi_plus_rate_hz','port_asymmetry','polarization_asymmetry'
]].agg(['mean','std'])
summary


total_singles_rate_hz                 \
                                        mean            std   
fault_class                                                   
attenuation_loss                8.940464e+05   79191.497266   
healthy                         9.593975e+05     979.645631   
polarization_drift              9.594086e+05     978.733475   
raman_noise                     1.562309e+06  801524.106846   
source_brightness_loss          8.689389e+05  103008.519544   
source_clock_drift              9.596822e+05    1025.242013   
spectral_detuning               9.595120e+05     993.273583   
sync_issue                      9.594001e+05     977.106803   
temperature_change              9.596910e+05    1029.975619   

                       accepted_bsm_rate_hz             psi_minus_rate_hz  \
                                       mean         std              mean   
fault_class                                                                 
attenuation_loss                1833.054815  376.231099        903.933453   
healthy                         2149.922272   46.373068       1059.895249   
polarization_drift              2149.942424   46.530911       1059.791524   
raman_noise                     2279.647765  199.022036       1124.556046   
source_brightness_loss          1732.146325  462.060162        854.408581   
source_clock_drift              1906.361791  332.107515        944.912421   
spectral_detuning               2149.719367   46.200540       1059.482013   
sync_issue                      1659.635194  492.481255        818.144299   
temperature_change              1887.792619  333.650177        936.030228   

                                   psi_plus_rate_hz              \
                               std             mean         std   
fault_class                                                       
attenuation_loss        186.134703       929.121362  192.559970   
healthy                  32.557830      1090.027023   32.990334   
polarization_drift       32.547428      1090.150900   33.029495   
raman_noise             102.255754      1155.091719  102.459107   
source_brightness_loss  227.928908       877.737744  235.993380   
source_clock_drift      161.508897       961.449370  173.565266   
spectral_detuning        34.325034      1090.237354   34.683176   
sync_issue              243.615371       841.490894  250.528457   
temperature_change      162.256346       951.762391  174.332357   

                       port_asymmetry           polarization_asymmetry  \
                                 mean       std                   mean   
fault_class                                                              
attenuation_loss        -2.237373e-07  0.001061          -3.870680e-06   
healthy                  1.801910e-07  0.001024          -1.342869e-07   
polarization_drift      -2.432878e-06  0.001018          -6.268047e-06   
raman_noise             -2.388117e-06  0.000887           1.228960e-07   
source_brightness_loss   5.870909e-06  0.001074           6.776569e-06   
source_clock_drift      -4.450274e-07  0.001019          -2.179048e-06   
spectral_detuning        7.588701e-07  0.001019          -7.006532e-08   
sync_issue               1.561705e-06  0.001017          -5.251143e-06   
temperature_change       5.865239e-06  0.001022           7.585068e-06   

                                  
                             std  
fault_class                       
attenuation_loss        0.001063  
healthy                 0.001021  
polarization_drift      0.001019  
raman_noise             0.000886  
source_brightness_loss  0.001080  
source_clock_drift      0.001019  
spectral_detuning       0.001019  
sync_issue              0.001022  
temperature_change      0.001017

In [6]:
joined = truth.merge(observable, on=['episode_id','window_index','fault_class'])
joined.groupby('fault_class')[['fault_progress','accepted_bsm_rate_hz','total_singles_rate_hz']].corr().iloc[0::3, -2:]


,,accepted_bsm_rate_hz,total_singles_rate_hz
fault_class,,,
attenuation_loss,fault_progress,-0.891361,-0.877663
healthy,fault_progress,NaN,NaN
polarization_drift,fault_progress,0.004484,-0.000735
raman_noise,fault_progress,0.701734,0.797059
source_brightness_loss,fault_progress,-0.938753,-0.917748
source_clock_drift,fault_progress,-0.806956,0.283576
spectral_detuning,fault_progress,0.001320,0.116746
sync_issue,fault_progress,-0.952547,-0.010054
temperature_change,fault_progress,-0.752535,0.265992
